In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPRegressor
from sklearn import metrics

In [2]:
np.random.seed(42)

r2_score_list = []
rmse_score_list = []
for i in range(10):
    data = pd.read_csv('data/data_DFT_mordredpca.csv')
    data['PC_ID'] = data.index // 14
    shuffled_groups = data['PC_ID'].unique()
    np.random.shuffle(shuffled_groups)
    train_groups = shuffled_groups[:10]
    test_groups = shuffled_groups[10:]
    train_data = data[data['PC_ID'].isin(train_groups)].reset_index(drop=True)
    test_data = data[data['PC_ID'].isin(test_groups)].reset_index(drop=True)

    y_train = pd.DataFrame(train_data['Yield'],columns=['Yield'])
    X_train = train_data.drop(columns=['PC_ID', 'Alc_ID', 'Yield', 'PC_SMILES', 'Alc_SMILES'])
    y_test = pd.DataFrame(test_data['Yield'],columns=['Yield'])
    X_test = test_data.drop(columns=['PC_ID', 'Alc_ID', 'Yield', 'PC_SMILES', 'Alc_SMILES'])
    
    a_X_train = (X_train - X_train.mean()) / X_train.std()
    a_X_test = (X_test - X_train.mean()) / X_train.std()
    a_X_train = a_X_train.dropna(how='any', axis=1)
    a_X_test = a_X_test[a_X_train.columns]
    param ={'hidden_layer_sizes':[(128,128,), (256,256,), (512,512,)], 'alpha':[0, 1, 2, 5]}
    reg = GridSearchCV(MLPRegressor(random_state=0, max_iter=1500, learning_rate_init=0.03),
                       param_grid=param, cv=5, n_jobs=12)
    reg.fit(a_X_train, y_train['Yield'])
    best = reg.best_estimator_
    #print(f'Run{i} model:', best)
    y_pred1 = best.predict(a_X_train)
    y_pred2 = best.predict(a_X_test)
    r2 = metrics.r2_score(y_test, y_pred2)
    rmse = metrics.root_mean_squared_error(y_test, y_pred2)
    print(f'Run{i} R2 (test):', r2, ', RMSE (test):', rmse)
    r2_score_list.append(r2)
    rmse_score_list.append(rmse)
print('==========(Result)==========')
print('Mean R2:', np.mean(r2_score_list))
print('SD R2:', np.std(r2_score_list))
print('Mean RMSE:', np.mean(rmse_score_list))
print('SD RMSE:', np.std(rmse_score_list))

Run0 R2 (test): -0.33552320435826966 , RMSE (test): 21.78358743645311
Run1 R2 (test): -0.7047906190741817 , RMSE (test): 29.33007549933832
Run2 R2 (test): 0.2120844570232231 , RMSE (test): 12.961399024870817
Run3 R2 (test): -1.1564811555132306 , RMSE (test): 35.351904604287064
Run4 R2 (test): -0.04473376358572012 , RMSE (test): 22.053591065159452
Run5 R2 (test): -2.881184247470888 , RMSE (test): 46.390431241923196
Run6 R2 (test): -4.477800264567191 , RMSE (test): 34.76013795650033
Run7 R2 (test): -0.04666067978160071 , RMSE (test): 24.864406288457324
Run8 R2 (test): -0.46554566413954346 , RMSE (test): 32.7139578467295
Run9 R2 (test): -0.8309076999990395 , RMSE (test): 35.06989665782132
==========(Result)==========
Mean R2: -1.0731542841466442
SD R2: 1.4049330773646955
Mean RMSE: 29.527938762154044
SD RMSE: 8.93633554046055
